# 📚 Notebook 01 — Coleta e Deduplicação de Dados

**Projeto:** Revisão Sistemática — IA na Educação  
**Protocolo:** PRISMA 2020 (`docs/protocol/prisma_protocol.md`)  
**Etapa:** 1 de 3 — Identificação e Triagem  

---

Este notebook executa as etapas de **coleta, deduplicação e triagem primária** dos artigos científicos coletados via API do Semantic Scholar.

**Fluxo:**
1. Configuração do ambiente
2. Carregamento dos dados brutos (`data/raw/search_results.json`)
3. Deduplicação por DOI e similaridade de título
4. Triagem por critérios PRISMA (inclusão/exclusão)
5. Exportação para `data/processed/papers_triados.csv`

In [ ]:
# ===========================================================================
# CÉLULA 1 — Configuração para execução no Google Colab
# ===========================================================================
# Descomente as linhas abaixo se estiver executando no Google Colab:

# from google.colab import drive
# drive.mount('/content/drive')
# import os
# os.chdir('/content/drive/MyDrive/ia_educacao_research')
# !pip install pandas requests matplotlib seaborn tqdm scikit-learn

print('✅ Configuração do ambiente concluída.')

In [ ]:
# ===========================================================================
# CÉLULA 2 — Importações
# ===========================================================================
import json
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime

print(f'pandas: {pd.__version__}')
print(f'numpy: {np.__version__}')

In [ ]:
# ===========================================================================
# CÉLULA 3 — Carregamento dos Dados Brutos
# ===========================================================================
RAW_DATA_PATH = Path('../data/raw/search_results.json')

if RAW_DATA_PATH.exists():
    with open(RAW_DATA_PATH, 'r', encoding='utf-8') as f:
        papers_raw = json.load(f)
    df_raw = pd.DataFrame(papers_raw)
    print(f'✅ Dados carregados: {len(df_raw)} registros')
    print(f'Colunas disponíveis: {list(df_raw.columns)}')
    display(df_raw.head(3))
else:
    print('⚠️  Arquivo de dados brutos não encontrado.')
    print('Execute primeiro: python scripts/data_collection.py')
    # Criar DataFrame de exemplo para desenvolvimento
    df_raw = pd.DataFrame({
        'paperId': ['abc123', 'def456', 'ghi789'],
        'title': ['AI in K-12 Education', 'Machine Learning for Assessment', 'AI Ethics in Classrooms'],
        'abstract': ['This study examines...', 'We propose a framework...', 'Ethical considerations...'],
        'year': [2022, 2023, 2024],
        'venue': ['Nature Education', 'Computers & Education', 'AI & Society'],
        'citationCount': [45, 12, 89]
    })
    print(f'📋 DataFrame de exemplo criado: {len(df_raw)} registros')

In [ ]:
# ===========================================================================
# CÉLULA 4 — Deduplicação
# ===========================================================================
print(f'Registros antes da deduplicação: {len(df_raw)}')

# Normalizar título para comparação
df_raw['title_normalized'] = (
    df_raw['title']
    .str.lower()
    .str.strip()
    .str.replace(r'[^a-z0-9\s]', '', regex=True)
)

# Remover duplicatas por título normalizado
df_deduped = df_raw.drop_duplicates(subset=['title_normalized'], keep='first').copy()
n_duplicates = len(df_raw) - len(df_deduped)

print(f'Duplicatas removidas: {n_duplicates}')
print(f'Registros após deduplicação: {len(df_deduped)}')

In [ ]:
# ===========================================================================
# CÉLULA 5 — Triagem PRISMA (Critérios de Inclusão/Exclusão)
# ===========================================================================

# Critério 1: Artigos com abstract
mask_has_abstract = df_deduped['abstract'].notna() & (df_deduped['abstract'] != '')

# Critério 2: Período 2020-2026
mask_year = df_deduped['year'].between(2020, 2026)

# Aplicar filtros
df_filtered = df_deduped[mask_has_abstract & mask_year].copy()

# Relatório de triagem
print('=' * 50)
print('RELATÓRIO DE TRIAGEM — PRISMA')
print('=' * 50)
print(f'  Registros brutos:           {len(df_raw)}')
print(f'  Após deduplicação:          {len(df_deduped)}')
print(f'  Excluídos sem abstract:     {(~mask_has_abstract).sum()}')
print(f'  Excluídos fora do período:  {(~mask_year).sum()}')
print(f'  Incluídos na síntese:       {len(df_filtered)}')
print('=' * 50)

In [ ]:
# ===========================================================================
# CÉLULA 6 — Exportação
# ===========================================================================
OUTPUT_PATH = Path('../data/processed/papers_triados.csv')
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

df_filtered.drop(columns=['title_normalized'], errors='ignore').to_csv(
    OUTPUT_PATH, index=False, encoding='utf-8'
)

print(f'✅ Dados exportados para: {OUTPUT_PATH}')
print(f'   Total de artigos triados: {len(df_filtered)}')
print(f'   Próximo passo: Execute o notebook 02_analise_textual.ipynb')